In [84]:
import pandas as pd
data = pd.read_csv('Churn_Modelling.csv')

In [85]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [86]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis = 1)

In [87]:
import pickle

In [88]:
with open ('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('ohe.pkl', 'rb') as file:
    ohe = pickle.load(file)

with open('scalar.pkl', 'rb') as file:
    scalar = pickle.load(file)

In [89]:
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


In [90]:
geo_encoded = ohe.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=ohe.get_feature_names_out(['Geography']))

In [91]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [92]:
X = data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [112]:
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,1,0.0,1.0,0.0


In [93]:
from sklearn.model_selection import train_test_split

In [94]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [95]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense

In [96]:
from tensorflow.keras import optimizers
model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer = 'adam', loss = 'mean_absolute_error', metrics=['mae'])

d:\Arjun\Study\Simple ANN Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [97]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [98]:
log_dir = "regressor/fit/" + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [99]:
early_stopping_callbacks = EarlyStopping(monitor = 'val_loss', patience=10, restore_best_weights=True)

In [100]:
history = model.fit(
    X_train,
    y_train,
    epochs = 100,
    validation_data = (X_test,y_test),
    callbacks = [early_stopping_callbacks,tensorboard_callback]
)

Epoch 1/100


250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 69377.6172 - mae: 69377.6172 - val_loss: 65680.0859 - val_mae: 65680.0859
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 62814.7969 - mae: 62814.7969 - val_loss: 57958.4766 - val_mae: 57958.4766
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 54220.2617 - mae: 54220.2617 - val_loss: 51476.4883 - val_mae: 51476.4883
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 51950.7695 - mae: 51950.7695 - val_loss: 50908.9609 - val_mae: 50908.9609
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 51376.9688 - mae: 51376.9688 - val_loss: 50482.3398 - val_mae: 50482.3398
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 51299.8594 - mae: 51299.8594 - val_loss: 52532.7852 - val_mae: 52532.7852
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 51363.3672 - mae: 51363.3672 - val_loss: 50367.0234 - val_mae: 50367.0234
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 51080.

In [101]:
model.save('regressor.h5')

In [102]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [103]:
tensorboard --logdir regressor/fit

Reusing TensorBoard on port 6006 (pid 3396), started 3:39:43 ago. (Use '!kill 3396' to kill it.)

In [123]:
input_data = {
    'CreditScore': 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance': 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'Exited' : 0,
}

In [124]:
pgeo_encoded = ohe.transform([[input_data['Geography']]]).toarray()
p_geo_endoded_df = pd.DataFrame(pgeo_encoded, columns=ohe.get_feature_names_out(['Geography']))

d:\Arjun\Study\Simple ANN Project\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [125]:
input_data_df = pd.DataFrame([input_data])

In [126]:
input_data_df['Gender'] = label_encoder_gender.fit_transform(input_data_df['Gender'])

In [127]:
input_data_df = pd.concat([input_data_df.drop('Geography',axis = 1),p_geo_endoded_df],axis=1)

In [128]:
input_data_scaled = scalar.fit_transform(input_data_df)

In [132]:
res = model.predict(input_data_scaled)
print(res[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
163.4966
